# Задание 3. Геометрический анализ сегментов

Раздел 6 ТЗ. Для каждого сегмента (отдельный `.obj`-файл) вычисляем набор дескрипторов:

- **6.1 Плотность точек** — среднее расстояние до соседей, равномерность и качественная оценка.
- **6.2 Форма распределения** — PCA: линейная / плоская / объёмная.
- **6.3 Структура поверхности** — локальная кривизна и её классификация (гладкая / слабо / сильно искривлённая / сложная).
- **6.4 Согласованность нормалей** — средний угол между нормалями соседних точек.
- **6.5 Связность** — число компонент, изолированные кластеры, целостность.
- **6.6 Сводка** — обобщённое описание сегмента.

На выходе: текстовый отчёт, CSV со всеми признаками и набор визуализаций.

## 1. Импорты и пути

In [ ]:
import csv
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.decomposition import PCA

# исходные сегменты (.obj) и каталог результатов (macOS)
SEGMENTS_DIR = Path("~/Datasets/lidar_pipe_fittings").expanduser()
REPORT_DIR = Path("outputs_task3")
REPORT_DIR.mkdir(exist_ok=True)

n_obj = len(list(SEGMENTS_DIR.glob("*.obj"))) if SEGMENTS_DIR.exists() else None
print("Найдено .obj:", n_obj if n_obj is not None else "директория не найдена")

## 2. Чтение вершин OBJ

In [ ]:
def read_obj_vertices(path):
    '''Извлекает только координаты вершин из OBJ (строки вида `v x y z`).'''
    verts = []
    with open(path, "r") as fh:
        for line in fh:
            if line.startswith("v "):
                parts = line.split()
                if len(parts) >= 4:
                    verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
    return np.asarray(verts, dtype=np.float64)

## 3. Геометрические дескрипторы (6.1–6.5)

In [ ]:
def density_descriptor(points, k=8):
    '''6.1: среднее расстояние до соседей, равномерность и качественная оценка.'''
    if len(points) < 2:
        return None, None, "insufficient points"
    k_eff = min(k, len(points) - 1)
    tree = cKDTree(points)
    dists, _ = tree.query(points, k=k_eff + 1)
    nn = dists[:, 1:]
    mean_d = float(nn.mean())
    uniformity = float(nn.std() / (mean_d + 1e-12))     # 0 -> идеально равномерно

    diag = float(np.linalg.norm(np.ptp(points, axis=0)))
    rel = mean_d / (diag + 1e-12)
    if rel < 0.01:
        quality = "high"
    elif rel > 0.05:
        quality = "low"
    else:
        quality = "medium"
    return mean_d, uniformity, quality


def shape_descriptor(points):
    '''6.2: PCA -> элонгация и тип формы.'''
    if len(points) < 3:
        return 0.0, np.array([1, 0, 0.0]), "insufficient"
    pca = PCA(n_components=3).fit(points)
    evals = pca.explained_variance_
    if len(evals) < 3:
        evals = np.pad(evals, (0, 3 - len(evals)), constant_values=0)
    elongation = float(evals[0] / (evals[2] + 1e-9))
    share = evals / (evals.sum() + 1e-9)
    if share[0] > 0.8:
        shape = "linear"            # одна доминирующая ось
    elif (share[0] + share[1]) > 0.9 and share[2] < 0.1:
        shape = "planar"            # две доминирующие оси
    else:
        shape = "volumetric"
    return elongation, pca.components_[0], shape


def curvature_descriptor(points, k=8):
    '''6.3: локальная кривизна (surface variation = lam3 / sum(lam)) в k-окрестности.'''
    if len(points) < k + 1:
        return None, None, "too few points"
    tree = cKDTree(points)
    _, idx = tree.query(points, k=k + 1)
    curv = []
    for i in range(len(points)):
        ev = np.sort(np.linalg.eigvalsh(np.cov(points[idx[i, 1:]].T)))[::-1]
        ev = np.maximum(ev, 1e-12)
        curv.append(ev[2] / ev.sum())
    curv = np.asarray(curv)
    mean_c, std_c = float(curv.mean()), float(curv.std())
    if mean_c < 0.02:
        cls = "smooth"
    elif mean_c < 0.08:
        cls = "slightly curved"
    elif mean_c < 0.20:
        cls = "highly curved"
    else:
        cls = "complex (combined)"
    return mean_c, std_c, cls


def normal_alignment(points, k=8):
    '''6.4: средний угол между нормалями соседних точек.

    Нормаль точки — собственный вектор с наименьшим собственным значением
    ковариации её окрестности.'''
    if len(points) < k + 1:
        return None, "insufficient points"
    tree = cKDTree(points)
    _, idx = tree.query(points, k=k + 1)
    normals = np.zeros((len(points), 3))
    for i in range(len(points)):
        _, evec = np.linalg.eigh(np.cov(points[idx[i, 1:]].T))
        normals[i] = evec[:, 0]
    # берём |cos|, чтобы не зависеть от ориентации нормали
    angles = []
    for i in range(len(points)):
        for j in idx[i, 1:]:
            cos = max(-1.0, min(1.0, abs(float(normals[i] @ normals[j]))))
            angles.append(np.arccos(cos))
    mean_ang = float(np.mean(angles))
    if mean_ang < 0.2:          # < ~11 град.
        quality = "high (planar/smooth)"
    elif mean_ang < 0.5:        # < ~28 град.
        quality = "medium (cylindrical/spherical)"
    else:
        quality = "low (complex/noisy)"
    return mean_ang, quality


def connectivity_descriptor(points, threshold_factor=2.5):
    '''6.5: число компонент связности и целостность.

    Порог расстояния = threshold_factor * среднее ближайшее расстояние.'''
    if len(points) < 2:
        return 0, 0, 0.0, "no points"
    tree = cKDTree(points)
    d_nn, _ = tree.query(points, k=2)
    radius = float(d_nn[:, 1].mean()) * threshold_factor

    parent = list(range(len(points)))

    def root(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for a, b in tree.query_pairs(r=radius):
        ra, rb = root(a), root(b)
        if ra != rb:
            parent[ra] = rb

    sizes = Counter(root(i) for i in range(len(points)))
    n_components = len(sizes)
    isolated = sum(1 for s in sizes.values() if s <= 3)
    integrity = float(max(sizes.values()) / len(points))
    if integrity > 0.99:
        integrity_quality = "highly intact (one main component)"
    elif integrity > 0.8:
        integrity_quality = "mostly intact with small outliers"
    else:
        integrity_quality = "fragmented or many components"
    return n_components, isolated, integrity, integrity_quality

## 4. Полное описание сегмента (6.6 — сводка)

In [ ]:
def describe_segment(points, verbose=False):
    '''Собирает все дескрипторы сегмента в одну запись + интерпретацию.'''
    mean_d, uniformity, dens_q = density_descriptor(points)
    elong, _, shape_type = shape_descriptor(points)
    mean_c, std_c, surf_cls = curvature_descriptor(points)
    mean_ang, cons_q = normal_alignment(points)
    n_comp, isolated, integrity, integ_q = connectivity_descriptor(points)

    record = {
        "num_points": int(len(points)),
        # 6.1
        "mean_neighbor_dist": mean_d, "uniformity": uniformity, "density_qual": dens_q,
        # 6.2
        "elongation": elong, "shape_type": shape_type,
        # 6.3
        "mean_curvature": mean_c, "curvature_std": std_c, "surface_class": surf_cls,
        # 6.4
        "mean_normal_angle_deg": float(np.degrees(mean_ang)) if mean_ang is not None else None,
        "normals_consistency": cons_q,
        # 6.5
        "n_components": n_comp, "isolated_clusters": isolated,
        "integrity": integrity, "integrity_qual": integ_q,
    }
    if verbose:
        print(f'  Точек: {record["num_points"]}')
        print(f"  6.1 плотность: mean_nn={mean_d:.4f}, uniformity={uniformity:.3f}, qual={dens_q}")
        print(f"  6.2 форма:     elongation={elong:.2f}, type={shape_type}")
        print(f"  6.3 поверхн.:  curvature={mean_c:.4f}±{std_c:.4f}, class={surf_cls}")
        if mean_ang is not None:
            print(f"  6.4 нормали:   angle={np.degrees(mean_ang):.1f} deg, qual={cons_q}")
        print(f"  6.5 связность: components={n_comp}, isolated={isolated}, "
              f"integrity={integrity:.2f} ({integ_q})")
    return record

## 5. Обработка всей папки и выгрузка отчётов

In [ ]:
obj_files = sorted(SEGMENTS_DIR.glob("*.obj"))
print(f"Анализируем {len(obj_files)} OBJ-файлов")

records = []
for path in obj_files:
    pts = read_obj_vertices(path)
    if len(pts) == 0:
        print(f"  пропуск (пусто): {path.name}")
        continue
    print(f"\n>>> {path.name}")
    rec = describe_segment(pts, verbose=True)
    rec["file"] = path.name
    records.append(rec)

columns = [
    "file", "num_points",
    "density_qual", "mean_neighbor_dist", "uniformity",
    "shape_type", "elongation",
    "surface_class", "mean_curvature", "curvature_std",
    "normals_consistency", "mean_normal_angle_deg",
    "n_components", "isolated_clusters", "integrity", "integrity_qual",
]
csv_path = REPORT_DIR / "segments_analysis_report.csv"
if records:
    with open(csv_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns)
        writer.writeheader()
        for r in records:
            writer.writerow({c: r.get(c) for c in columns})
    print("\nCSV-отчёт:", csv_path)

txt_path = REPORT_DIR / "segments_analysis_report.txt"
with open(txt_path, "w") as fh:
    for r in records:
        fh.write(f'=== {r["file"]} ===\n')
        fh.write(f'  Точек: {r["num_points"]}\n')
        fh.write(f'  6.1 плотность: mean_nn={r["mean_neighbor_dist"]:.4f}, qual={r["density_qual"]}\n')
        fh.write(f'  6.2 форма:     elongation={r["elongation"]:.2f}, type={r["shape_type"]}\n')
        fh.write(f'  6.3 поверхн.:  curvature={r["mean_curvature"]:.4f}, class={r["surface_class"]}\n')
        if r["mean_normal_angle_deg"] is not None:
            fh.write(f'  6.4 нормали:   angle={r["mean_normal_angle_deg"]:.1f} deg, '
                     f'qual={r["normals_consistency"]}\n')
        fh.write(f'  6.5 связность: components={r["n_components"]}, '
                 f'integrity={r["integrity"]:.2f} ({r["integrity_qual"]})\n\n')
print("Текстовый отчёт:", txt_path)

## 6. Визуализация: распределения признаков

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

shape_counts = Counter(r["shape_type"] for r in records)
axes[0, 0].bar(shape_counts.keys(), shape_counts.values(), color="#2563eb")
axes[0, 0].set_title("6.2 Типы формы")
axes[0, 0].set_ylabel("Кол-во")

surface_counts = Counter(r["surface_class"] for r in records)
axes[0, 1].bar(surface_counts.keys(), surface_counts.values(), color="#16a34a")
axes[0, 1].set_title("6.3 Классы поверхности")
for t in axes[0, 1].get_xticklabels():
    t.set_rotation(30)
    t.set_ha("right")

curv_vals = [r["mean_curvature"] for r in records if r["mean_curvature"] is not None]
axes[1, 0].hist(curv_vals, bins=20, color="#f59e0b", edgecolor="black")
axes[1, 0].set_title("6.3 Средняя кривизна")
axes[1, 0].set_xlabel("mean curvature")
axes[1, 0].set_ylabel("Кол-во сегментов")

angle_vals = [r["mean_normal_angle_deg"] for r in records if r["mean_normal_angle_deg"] is not None]
axes[1, 1].hist(angle_vals, bins=20, color="#dc2626", edgecolor="black")
axes[1, 1].set_title("6.4 Угол между нормалями (deg)")
axes[1, 1].set_xlabel("degrees")
axes[1, 1].set_ylabel("Кол-во сегментов")

plt.tight_layout()
plt.savefig(REPORT_DIR / "feature_distributions.png", dpi=120)
plt.show()

## 7. Сводная таблица: классы поверхности по типам формы

In [ ]:
pivot = defaultdict(lambda: defaultdict(int))
for r in records:
    pivot[r["shape_type"]][r["surface_class"]] += 1

shapes = sorted(pivot.keys())
surface_classes = sorted({c for d in pivot.values() for c in d})

print(f'{"shape \\ surface":>20s} | ' + " | ".join(f"{c:>20s}" for c in surface_classes))
print("-" * (22 + 22 * len(surface_classes)))
for s in shapes:
    line = " | ".join(f"{pivot[s].get(c, 0):>20d}" for c in surface_classes)
    print(f"{s:>20s} | {line}")

## 8. Визуализация сегмента: точки с подсветкой кривизны

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

sample_path = obj_files[0]
pts = read_obj_vertices(sample_path)

tree = cKDTree(pts)
_, idx = tree.query(pts, k=9)
point_curv = []
for i in range(len(pts)):
    ev = np.sort(np.linalg.eigvalsh(np.cov(pts[idx[i, 1:]].T)))[::-1]
    ev = np.maximum(ev, 1e-12)
    point_curv.append(ev[2] / ev.sum())
point_curv = np.array(point_curv)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=point_curv, cmap="viridis", s=2)
ax.set_title(f"{sample_path.name}\nЛокальная кривизна (surface variation)")
ax.set_axis_off()
fig.colorbar(sc, ax=ax, shrink=0.6, label="lam3 / sum(lam)")
plt.tight_layout()
plt.savefig(REPORT_DIR / "sample_curvature.png", dpi=120)
plt.show()

## 9. Выводы

1. По разделу 6 ТЗ реализованы все пять геометрических метрик: плотность, форма распределения,
   структура поверхности, согласованность нормалей и топологическая связность.
2. Все дескрипторы строятся на стандартных подходах: PCA по координатам и kNN-окрестностям,
   собственные значения ковариации (Demantke et al., 2011), union-find по графу соседей.
3. Для каждого OBJ-файла формируется единая запись с признаками и качественной интерпретацией;
   результат сохраняется в CSV и в человекочитаемый текстовый отчёт.
4. Гистограммы признаков и сводная таблица показывают, как распределены формы и типы
   поверхностей по датасету. Эти данные удобно использовать как **вход для выбора метода
   реконструкции** в Задании 2 (plane -> Alpha Shape, tube -> Ball Pivoting, sphere/complex -> Poisson).
5. Подсветка точечной кривизны (раздел 8) выделяет места резких изломов и подтверждает
   корректность вычисляемых дескрипторов.